# Classificação Supervisionada de Uso e Cobertura do Solo
**Área de Foco:** Região Metropolitana de Belém (PA) – Entorno do Lago da Água Preta e Rio Guamá.

### Objetivo
Este notebook é um projeto de estudo focado na transição do ambiente Google Earth Engine (JavaScript) para o ecossistema Python local. O objetivo central não foi a produção de um mapa de uso do solo com rigor científico, mas sim a compreensão técnica das bibliotecas geoespaciais e o teste de fluxo de dados (pipelines) para classificação supervisionada. 

Teve como objetivo realizar a classificação automatizada da cobertura do solo em uma área específica de Belém, diferenciando entre três classes principais: Água, Vegetação e Área Urbana

### Tecnologias e Algoritmos
Para o processamento e análise, foram utilizadas as seguintes ferramentas:
* **Bibliotecas:** `geemap` para interface cartográfica e `ee` (Earth Engine API) para processamento em nuvem.
* **Dados:** Imagens de refletância de superfície do satélite **Sentinel-2 (Copernicus)**.
* **Algoritmos de Classificação:**
    * **CART (Classification and Regression Trees):** Algoritmo de árvore de decisão simples.
    * **Random Forest:** Conjunto de árvores de decisão (floresta aleatória), utilizado para obter maior precisão e estabilidade nos resultados.

###  Dificuldades Encontradas 
O desenvolvimento deste ambiente local apresentou desafios técnicos superados ao longo do processo:
* **Configuração de Ambiente:** Sincronizar as versões do Python e Conda com as bibliotecas geoespaciais (`gdal`, `rasterio`) no Windows, evitando conflitos de DLLs.
* **Autenticação:** Superar instabilidades nos protocolos de autenticação (Erros HTTP 500) através do uso do modo manual via terminal (`auth_mode='notebook'`).
* **Estrutura de Dados:** A correção crítica da geometria, transformando coleções de *MultiPoints* em *Features* individuais para que os algoritmos tivessem volume de dados (amostras) suficiente para o treinamento (evitando o erro de "No valid training data").

---

In [ ]:
import ee
import geemap

In [2]:
# Roda o comando no terminal : earthengine authenticate --auth_mode=notebook
try:
    ee.Initialize(project='spatial-yew-490017-r3')
    print("Sucesso: Já estava autenticado!")
except Exception as e:
    print("Iniciando processo de autenticação...")
    ee.Authenticate(auth_mode='notebook')
    ee.Initialize(project='spatial-yew-490017-r3')
    print("Autenticado com sucesso!")
    
    

Sucesso: Já estava autenticado!


In [3]:
# Área de Estudo  - Belém, Para, Águas Lindas/Lago da água Preta
geometry = ee.Geometry.MultiPolygon([
    [[[-48.43487450457067, -1.3979174700709287], [-48.43487450457067, -1.430866412087666], [-48.3882684407279, -1.430866412087666], [-48.3882684407279, -1.3979174700709287]]],
    [[[-48.393332451348016, -1.4310380199311954], [-48.393332451348016, -1.4316386472824194], [-48.39316078997106, -1.4316386472824194], [-48.39316078997106, -1.4310380199311954]]],
    [[[-48.39298912859411, -1.4318960589562155], [-48.39298912859411, -1.4318960589562155], [-48.39290329790563, -1.4318960589562155], [-48.39290329790563, -1.4318960589562155]]]]
)

# Lista de Features individuais em vez de MultiPoint
def create_points(coords, classe, label):
    return [ee.Feature(ee.Geometry.Point(coord), {'classe': classe, 'label': label}) for coord in coords]

# Coordenadas originais
coords_agua = [[-48.416256678371894, -1.4073049866029377], [-48.416771662502754, -1.4089781795845302], [-48.41737247732209, -1.4114665156417374], [-48.416771662502754, -1.4147699921545716], [-48.41702915456818, -1.4182021705618535], [-48.41707206991242, -1.421763050279315], [-48.433733952312934, -1.421258950170814], [-48.43251086500214, -1.4214305587307443], [-48.415527117519844, -1.424122666344248], [-48.415484202175605, -1.4261819656631947], [-48.40861774709748, -1.4257100430652663], [-48.4061715724759, -1.4203472794823995], [-48.40565658834504, -1.4173441264367377], [-48.405484926968086, -1.4139119467588752], [-48.40488411214875, -1.410994590042256], [-48.39186584094567, -1.4267315174891855], [-48.39180817345185, -1.4261509991911054], [-48.39160298446221, -1.425944533063629], [-48.432432735562564, -1.4210567388428048], [-48.4318265563252, -1.417469044155201], [-48.430710757375, -1.415913838345206], [-48.426614242004, -1.4154044744188343], [-48.42437727968558, -1.4145625176074759]]
coords_veg = [[-48.411584154824254, -1.4120600578164455], [-48.41235663102054, -1.4172941339702054], [-48.4114124934473, -1.4206405043307593], [-48.40892340348148, -1.4195250480819008], [-48.40892340348148, -1.4153206312336262], [-48.40394522354984, -1.4060537263983268], [-48.4004261653223, -1.4051098729115146], [-48.399567858437536, -1.4093143082328836], [-48.400855318764684, -1.4164360895086379], [-48.40197111771488, -1.4239010656610052], [-48.4059193293848, -1.4283628789525447], [-48.41372992203617, -1.4329104874304044], [-48.42711950943851, -1.429649938870111], [-48.42548872635746, -1.4233004362885133], [-48.414536531302765, -1.4008729384192513], [-48.42337354383323, -1.4029700690713836], [-48.39765411489762, -1.4163819065715193], [-48.40147358053483, -1.4103005072132755], [-48.40181690328873, -1.407383145968399], [-48.39885574453629, -1.4208544599394715]]
coords_urb = [[-48.396023331816565, -1.411737735305948], [-48.39595895880021, -1.4111800051067729], [-48.39613062017716, -1.4097320510808635], [-48.3985875236348, -1.416682222176233], [-48.39908105009354, -1.4199749657157361], [-48.42616982061571, -1.4103792451700157], [-48.42438883382982, -1.4101647334455159], [-48.42302627165026, -1.4077943775740975], [-48.42347688276476, -1.4067003663588509], [-48.42021531660265, -1.4052738607685], [-48.417361446210805, -1.4045981472899758], [-48.427465242523176, -1.406291132342058], [-48.42659620680235, -1.406248229923481], [-48.42594174780272, -1.4056154191578751], [-48.42040096257828, -1.4261223237958571], [-48.418968662964325, -1.4269857274116713], [-48.408287492483225, -1.4038855420230374], [-48.40795489856538, -1.4051726155395996], [-48.40259048053559, -1.4020675494737662], [-48.39666946514774, -1.4009424461114306], [-48.39715226277042, -1.3998216176046183], [-48.40970176100212, -1.403607869924518], [-48.39299152915134, -1.4306644522584944]]

# Criando as coleções
features_agua = create_points(coords_agua, 0, 'agua')
features_veg = create_points(coords_veg, 1, 'vegetacao')
features_urb = create_points(coords_urb, 2, 'urbana')

legend_dict = {
    'Agua': '0000FF',
    'Vegetacao': '008000',
    'Urbano': 'FF0000'
}

In [4]:
def style_points(feature):
    color = ee.Dictionary({
        '0': 'blue',
        '1': 'green',
        '2': 'red'
    }).get(ee.Number(feature.get('classe')).format())
    return feature.set({'style': {'color': color, 'pointShape': 'circle', 'pointSize': 3}})

In [5]:
# Coleção de amostras para facilitar a visualização
collections = ee.FeatureCollection(features_agua + features_veg + features_urb)
collections = collections.map(style_points)
collections

In [6]:
# Criar a instância do mapa
Map = geemap.Map()
Map.add_basemap('SATELLITE')
Map.centerObject(geometry, 14)

# Adicionando a área de estudo
Map.addLayer(ee.Image().paint(geometry, 0, 3), {'palette': ['FFFF00']}, 'Área de Estudo')
Map.addLayer(collections.style(styleProperty='style'), {}, 'Amostras')

# Removemos controles antigos 
if hasattr(Map, 'legend_dict'):
    Map.remove_legend()
    
# Adiciona uma legenda para facilitar a leitura
Map.add_legend(title="Classes", legend_dict=legend_dict)

Map

Map(center=[-1.4143931567372174, -48.41157024238487], controls=(WidgetControl(options=['position', 'transparen…

In [7]:
# Processamento da Imagem Sentinel-2
s2_img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .select("B.*")
          .filterBounds(geometry)
          .filterDate("2024-01-01", "2024-09-01")
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
          .sort('CLOUDY_PIXEL_PERCENTAGE')
          .first())

s2_clip = s2_img.clip(geometry)
bands = s2_clip.bandNames() 

print("Bandas disponíveis:", bands.getInfo())
print("Informações da imagem:", s2_clip.getInfo())
print("collections features:", collections.size().getInfo())

Bandas disponíveis: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']
Informações da imagem: {'type': 'Image', 'bands': [{'id': 'B1', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [88, 64], 'origin': [1424, 911], 'crs': 'EPSG:32722', 'crs_transform': [60, 0, 699960, 0, -60, 9900040]}, {'id': 'B2', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [520, 378], 'origin': [8548, 5470], 'crs': 'EPSG:32722', 'crs_transform': [10, 0, 699960, 0, -10, 9900040]}, {'id': 'B3', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [520, 378], 'origin': [8548, 5470], 'crs': 'EPSG:32722', 'crs_transform': [10, 0, 699960, 0, -10, 9900040]}, {'id': 'B4', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 65535}, 'dimensions': [520, 378], 'origin': [8548, 5470], 'crs': 'EPSG:32722', 'crs_transform': [10, 0, 699960, 0, -10, 990

In [8]:

training = s2_clip.sampleRegions(
    collection=collections, 
    properties=['classe'], 
    scale=20,
    geometries=True
)

# Treinar o Classificador CART (smileCart)
trained_cart = ee.Classifier.smileCart().train(
    features=training, 
    classProperty='classe', 
    inputProperties=bands
)

# Treinar o Classificador Random Forest 
trained_rf = ee.Classifier.smileRandomForest(10).train(
    features=training, 
    classProperty='classe', 
    inputProperties=bands
)

In [9]:
# Aplicar as classificações na imagem recortada
classified_cart = s2_clip.classify(trained_cart)
classified_rf = s2_clip.classify(trained_rf)

class_vis = {
    'min': 0, 
    'max': 2, 
    'palette': ['blue', 'green', 'red']
}

# Criar o mapa de resultados
Map = geemap.Map()
Map.centerObject(geometry, 14)
Map.add_basemap('SATELLITE')

# Adicionar as camadas de classificação
Map.addLayer(classified_cart, class_vis, 'Classificação CART')
Map.addLayer(classified_rf, class_vis, 'Classificação Random Forest')

Map

Map(center=[-1.4143931567372174, -48.41157024238487], controls=(WidgetControl(options=['position', 'transparen…